# Misinformation Models—
Trains all models to date and saves them.

Claude helped on implementation details. Logic and concepts are human generated.

### Import configuration and packages ###

In [1]:
##TO RUN ON COLAB ONLY

#installs
!pip install tweet-preprocessor==0.5.0 feedparser whoosh iterative-stratification fastapi uvicorn
!pip install nltk spacy

#imports
import sys
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import torch
import joblib, pickle, torch.nn as nn

#mount google drive and set path-related variables.
drive.mount('/content/drive')
BASE_DIR=Path("/content/drive/MyDrive/linguistic_markers")
SPRINT_DIR = BASE_DIR / "581_Sprint_4"
SRC_DIR = SPRINT_DIR / "src"
DATA_DIR = BASE_DIR / "data" / "final_splits"
SAVE_DIR = SPRINT_DIR / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Add the SPRINT_DIR to the system path so Python can find modules like 'cnn_baseline'
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(1, str(SPRINT_DIR))
sys.path.insert(2, str(SRC_DIR))
sys.path.insert(3, str(DATA_DIR))
sys.path.insert(4, str(BASE_DIR))

# Define the device for training
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


import cnn_baseline as cnn
#Running this will import FastText vector file, stored on HuggingFace and is >4gb.
from config import FASTTEXT_PATH, TARGETS, SEED
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report
import random
import builtins
builtins.Path = Path
import cnn_mtl_no_ling as cnn_mtl_pos

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 23.6 MB/s eta 0:00:00
  Created wheel for tweet-preprocessor: filename=tweet_preprocessor-0.5.0-py3-none-any.whl size=7928 sha256=667387891e79df88b545f7b99544c162d0a29e6d63c281bedb4953ca26a93b42
  Stored in directory: /root/.cache/pip/wheels/4b/6e/04/d26d41ed041dd0318b112367564452d05a4835d1a3f8e37518
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=0f22d7bfd094ad206b38fb1c376f0b9879aaf794ff23a516e0b5cc0cad3f04ea
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built tweet-preprocessor sgmllib3k
Mounted at /content/drive
Using device: cuda:0


cc.en.300.vec:   0%|          | 0.00/4.51G [00:00<?, ?B/s]

### Define model name constants and results lists


In [2]:
# Model name constants — define once, use everywhere
M_TEXT_CNN                 = "TextCNN"
M_TEXT_CNN_TRANSFER        = "TextCNN Transfer"
M_TEXT_CNN_TRANSFER_SHARED = "TextCNN_Transfer_shared"
M_CNN_MTL_LING             = "TextCNN MTL (POS + linguistic)"
M_CNN_MTL_POS              = "TextCNN MTL (POS)"
M_TEXT_CNN_BOOTSTRAP       = "TextCNN Bootstrap"
M_TEXT_CNN_FSL             = "TextCNN FSL"
M_LOGREG                   = "LogReg"
M_LOGREG_EMBED             = "LogReg (+ embeddings)"
M_LOGREG_MTL_POS           = "LogReg MTL (cascaded POS)"
M_LOGREG_EMBED_BOOTSTRAP   = "LogReg (+ embeddings) Bootstrap"
M_LOGREG_EMBED_FSL         = "LogReg (+ embeddings) FSL"
M_ENSEMBLE_SOFT_VOTE       = "Soft Vote Ensemble"
M_ENSEMBLE_MOTIVATED       = "Motivated Ensemble"


AL_PCTS        = [0.05, 0.10, 0.15, 0.20]
CNN_AL_MODELS  = [f"TextCNN AL {int(p*100)}%" for p in AL_PCTS]
LR_AL_MODELS   = [f"LogReg (+emb) AL {int(p*100)}%" for p in AL_PCTS]

CNN_MODELS  = [M_TEXT_CNN, M_TEXT_CNN_TRANSFER, M_CNN_MTL_LING, M_CNN_MTL_POS,
               M_TEXT_CNN_BOOTSTRAP, M_TEXT_CNN_FSL] + CNN_AL_MODELS
LR_MODELS   = [M_LOGREG, M_LOGREG_EMBED, M_LOGREG_MTL_POS,
               M_LOGREG_EMBED_BOOTSTRAP, M_LOGREG_EMBED_FSL] + LR_AL_MODELS
BASE_MODELS = LR_MODELS + CNN_MODELS
ENSEMBLE_MODELS = [M_ENSEMBLE_SOFT_VOTE, M_ENSEMBLE_MOTIVATED]

OPINION_LABEL = "opinion_label"
MISINFORM_LABEL = "misinformation_label"

# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

#metrics_cache for later use in ordering all models by Macro F1 and making easy displays
metrics_cache = {}

### Load data

In [3]:

# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


### Utility functions for saving check points, reproducibiity, displaying metrics, ranking metrics

In [4]:

def rank_key(metrics_key):
    """Sort key for model selection: primary = Macro F1, tiebreaker = AUC-ROC.
    Returns (-inf, -inf) for models not trained on this target, excluding them from selection."""
    m = metrics_cache.get(metrics_key)
    if m is None:
        return (float("-inf"), float("-inf"))
    return (m["macro_f1"], m.get("auc_roc", 0.0))


def display_metrics(key_prefix):
    """Display a metrics table for one ensemble/model prefix across all TARGETS."""
    pd.set_option("display.float_format", "{:.4f}".format)
    rows = []
    for t in TARGETS:
        key = f"{key_prefix} — {t}"
        m = metrics_cache[key]
        rows.append({
            "Model": key,
            "Macro F1": m["macro_f1"],
            "F1 (not-op)": m["f1_class0"],
            "F1 (opinion)": m["f1_class1"],
            "F1.5 (recall-weighted)": m["fbeta_class1"],
            "AUC-ROC": m.get("auc_roc", float("nan")),
        })
    return pd.DataFrame(rows).set_index("Model")

def make_reproducible():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

def save_checkpoint(key, model=None, result=None):
    """Save model weights and/or (preds, labels, probs) result tuple."""
    safe_key = key.replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "").replace("+", "plus")
    saved = []

    if model is not None:
        if isinstance(model, nn.Module):
            torch.save(model.state_dict(), SAVE_DIR / f"{safe_key}.pt")
            saved.append("state_dict")
        else:
            joblib.dump(model, SAVE_DIR / f"{safe_key}.joblib")
            saved.append("joblib")

    if result is not None:
        with open(SAVE_DIR / f"{safe_key}_result.pkl", "wb") as f:
            pickle.dump(result, f)
        saved.append("result")

    if saved:
        print(f"  [saved] {safe_key} ({', '.join(saved)})")


### Create and train models


#### CNN Baseline

In [5]:
#CNN Baseline

make_reproducible()

vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)


cnn_model_opinion = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model_opinion = cnn.train_model(cnn_model_opinion, train_loader, dev_loader, train_rows, DEVICE,
                                    train_targets=[OPINION_LABEL])
key = f"{M_TEXT_CNN} — {OPINION_LABEL}"
results[key] = cnn.predict(cnn_model_opinion, dev_loader, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_model_opinion, result=results[key])

cnn_model_misinform = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model_misinform = cnn.train_model(cnn_model_misinform, train_loader, dev_loader, train_rows, DEVICE,
                                      train_targets=[MISINFORM_LABEL])
key = f"{M_TEXT_CNN} — {MISINFORM_LABEL}"
results[key] = cnn.predict(cnn_model_misinform, dev_loader, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_model_misinform, result=results[key])


Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3907/4088 vocab tokens found in FastText vectors (95.6%)
Epoch   1 | loss=0.7675 | avg_dev_f1=0.6700 (opinion_label: 0.6700)
Epoch   2 | loss=0.6532 | avg_dev_f1=0.6862 (opinion_label: 0.6862)
Epoch   3 | loss=0.6069 | avg_dev_f1=0.7238 (opinion_label: 0.7238)
Epoch   4 | loss=0.5540 | avg_dev_f1=0.7234 (opinion_label: 0.7234)
Epoch   5 | loss=0.4960 | avg_dev_f1=0.6800 (opinion_label: 0.6800)
Epoch   6 | loss=0.4604 | avg_dev_f1=0.7287 (opinion_label: 0.7287)
Epoch   7 | loss=0.3911 | avg_dev_f1=0.7196 (opinion_label: 0.7196)
Epoch   8 | loss=0.3478 | avg_dev_f1=0.7242 (opinion_label: 0.7242)
Epoch   9 | loss=0.3092 | avg_dev_f1=0.7410 (opinion_label: 0.7410)
Epoch  10 | loss=0.2668 | avg_dev_f1=0.7147 (opinion_label: 0.7147)
Epoch  11 | loss=0.2322 | avg_dev_f1=0.7339 (opinion_label: 0.7339)
Epoch  12 | loss=0.2030 | avg

#### CNN Transfer Learning

In [6]:
# CNN Transfer Learning
cnn_tf_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_tf_model = cnn.train_model(cnn_tf_model, train_loader, dev_loader, train_rows, DEVICE)
save_checkpoint(M_TEXT_CNN_TRANSFER_SHARED, model=cnn_tf_model)

for target in TARGETS:
    key = f"{M_TEXT_CNN_TRANSFER} — {target}"
    results[key] = cnn.predict(cnn_tf_model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

Epoch   1 | loss=1.7590 | avg_dev_f1=0.6967 (opinion_label: 0.7076 | misinformation_label: 0.6858)
Epoch   2 | loss=1.5097 | avg_dev_f1=0.6910 (opinion_label: 0.6922 | misinformation_label: 0.6898)
Epoch   3 | loss=1.3458 | avg_dev_f1=0.7262 (opinion_label: 0.6889 | misinformation_label: 0.7635)
Epoch   4 | loss=1.1901 | avg_dev_f1=0.7498 (opinion_label: 0.7177 | misinformation_label: 0.7818)
Epoch   5 | loss=1.0373 | avg_dev_f1=0.7580 (opinion_label: 0.7033 | misinformation_label: 0.8128)
Epoch   6 | loss=0.9232 | avg_dev_f1=0.7640 (opinion_label: 0.7076 | misinformation_label: 0.8203)
Epoch   7 | loss=0.7993 | avg_dev_f1=0.7687 (opinion_label: 0.7129 | misinformation_label: 0.8245)
Epoch   8 | loss=0.7516 | avg_dev_f1=0.7708 (opinion_label: 0.7134 | misinformation_label: 0.8282)
Epoch   9 | loss=0.6607 | avg_dev_f1=0.7872 (opinion_label: 0.7134 | misinformation_label: 0.8609)
Epoch  10 | loss=0.5938 | avg_dev_f1=0.7769 (opinion_label: 0.7023 | misinformation_label: 0.8515)
Epoch  11 

#### CNN MTL — POS & Linguistic Features


In [7]:
import cnn_mtl_ling as cnn_mtl_ling

train_loader_ling = cnn_mtl_ling.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_ling   = cnn_mtl_ling.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

cnn_mtl_l_model_opinion = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_l_model_opinion = cnn_mtl_ling.train_model(cnn_mtl_l_model_opinion, train_loader_ling,
                                                    dev_loader_ling, train_rows, DEVICE,
                                                    train_targets=[OPINION_LABEL])
key = f"{M_CNN_MTL_LING} — {OPINION_LABEL}"
results[key] = cnn_mtl_ling.predict(cnn_mtl_l_model_opinion, dev_loader_ling, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_l_model_opinion, result=results[key])

cnn_mtl_l_model_misinform = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_l_model_misinform = cnn_mtl_ling.train_model(cnn_mtl_l_model_misinform, train_loader_ling,
                                                      dev_loader_ling, train_rows, DEVICE,
                                                      train_targets=[MISINFORM_LABEL])
key = f"{M_CNN_MTL_LING} — {MISINFORM_LABEL}"
results[key] = cnn_mtl_ling.predict(cnn_mtl_l_model_misinform, dev_loader_ling, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_l_model_misinform, result=results[key])


Epoch   1 | loss=0.7583 | avg_dev_f1=0.6987 (opinion_label: 0.6987)
Epoch   2 | loss=0.6579 | avg_dev_f1=0.6922 (opinion_label: 0.6922)
Epoch   3 | loss=0.6055 | avg_dev_f1=0.6648 (opinion_label: 0.6648)
Epoch   4 | loss=0.5716 | avg_dev_f1=0.6944 (opinion_label: 0.6944)
Epoch   5 | loss=0.5110 | avg_dev_f1=0.6750 (opinion_label: 0.6750)
Epoch   6 | loss=0.4479 | avg_dev_f1=0.7374 (opinion_label: 0.7374)
Epoch   7 | loss=0.4070 | avg_dev_f1=0.6946 (opinion_label: 0.6946)
Epoch   8 | loss=0.3581 | avg_dev_f1=0.7230 (opinion_label: 0.7230)
Epoch   9 | loss=0.2974 | avg_dev_f1=0.7190 (opinion_label: 0.7190)
Epoch  10 | loss=0.2555 | avg_dev_f1=0.7193 (opinion_label: 0.7193)
Epoch  11 | loss=0.2285 | avg_dev_f1=0.7190 (opinion_label: 0.7190)
  Early stopping (no improvement for 5 epochs). Best F1: 0.7374
  [saved] TextCNN_MTL_POS_plus_linguistic_—_opinion_label (state_dict, result)
Epoch   1 | loss=0.9857 | avg_dev_f1=0.5830 (misinformation_label: 0.5830)
Epoch   2 | loss=0.8269 | avg_dev_

#### CNN MTL - POS Features only


In [8]:
import cnn_mtl_no_ling as cnn_mtl_pos

train_loader_pos = cnn_mtl_pos.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_pos   = cnn_mtl_pos.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

cnn_mtl_p_model_opinion = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_p_model_opinion = cnn_mtl_pos.train_model(cnn_mtl_p_model_opinion, train_loader_pos,
                                                   dev_loader_pos, train_rows, DEVICE,
                                                   train_targets=[OPINION_LABEL])
key = f"{M_CNN_MTL_POS} — {OPINION_LABEL}"
results[key] = cnn_mtl_pos.predict(cnn_mtl_p_model_opinion, dev_loader_pos, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_p_model_opinion, result=results[key])

cnn_mtl_p_model_misinform = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_p_model_misinform = cnn_mtl_pos.train_model(cnn_mtl_p_model_misinform, train_loader_pos,
                                                     dev_loader_pos, train_rows, DEVICE,
                                                     train_targets=[MISINFORM_LABEL])
key = f"{M_CNN_MTL_POS} — {MISINFORM_LABEL}"
results[key] = cnn_mtl_pos.predict(cnn_mtl_p_model_misinform, dev_loader_pos, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_p_model_misinform, result=results[key])


Epoch   1 | loss=0.7599 | avg_dev_f1=0.6792 (opinion_label: 0.6792)
Epoch   2 | loss=0.6590 | avg_dev_f1=0.6928 (opinion_label: 0.6928)
Epoch   3 | loss=0.6039 | avg_dev_f1=0.7046 (opinion_label: 0.7046)
Epoch   4 | loss=0.5529 | avg_dev_f1=0.7037 (opinion_label: 0.7037)
Epoch   5 | loss=0.5071 | avg_dev_f1=0.7199 (opinion_label: 0.7199)
Epoch   6 | loss=0.4477 | avg_dev_f1=0.7081 (opinion_label: 0.7081)
Epoch   7 | loss=0.4074 | avg_dev_f1=0.7368 (opinion_label: 0.7368)
Epoch   8 | loss=0.3524 | avg_dev_f1=0.7383 (opinion_label: 0.7383)
Epoch   9 | loss=0.2995 | avg_dev_f1=0.7193 (opinion_label: 0.7193)
Epoch  10 | loss=0.2615 | avg_dev_f1=0.7246 (opinion_label: 0.7246)
Epoch  11 | loss=0.2199 | avg_dev_f1=0.7261 (opinion_label: 0.7261)
Epoch  12 | loss=0.2065 | avg_dev_f1=0.7368 (opinion_label: 0.7368)
Epoch  13 | loss=0.1686 | avg_dev_f1=0.7024 (opinion_label: 0.7024)
  Early stopping (no improvement for 5 epochs). Best F1: 0.7383
  [saved] TextCNN_MTL_POS_—_opinion_label (state_dic

#### Logistic Regression Baseline

In [9]:
import logreg_baseline as lr

for target in TARGETS:
    key = f"{M_LOGREG} — {target}"
    results[key] = lr.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013
  [saved] LogReg_—_opinion_label (result)

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8514
  [saved] LogReg_—_misinformation_label (result)


#### Logistic Regression Transfer Learning

In [10]:
import logreg_transfer as lr_transfer

for target in TARGETS:
    key = f"{M_LOGREG_EMBED} — {target}"
    results[key] = lr_transfer.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7285
  [saved] LogReg_plus_embeddings_—_opinion_label (result)

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8716
  [saved] LogReg_

#### LogReg MTL — Cascaded POS Prediction


In [11]:
import logreg_mtl as lr_mtl

for target in TARGETS:
    key = f"{M_LOGREG_MTL_POS} — {target}"
    results[key] = lr_mtl.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── LogReg MTL Cascaded POS  [opinion_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Running secondary task: POS tagging 600 documents …
  POS distribution feature dim: 15 tags
  Running secondary task: POS tagging 200 documents …
  POS distribution feature dim: 15 tags
  Feature matrix: train=(600, 2819), dev=(200, 2819)
  Running GridSearchCV over C
  Best C: 0.1  |  CV macro-F1: 0.7188
  [saved] LogReg_MTL_cascaded_POS_—_opinion_label (result)

── LogReg MTL Cascaded POS  [misinformation_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 

## Bootstrapping


*   Use TextCNN for Opinion
*   Use LogReg(+embeddings) for Misinformation

In [12]:
# Reproducibility — reset RNG state
make_reproducible()


# import 250 bootstrap rows with just id, text, and labels so predict function can be used
bootstrap_rows = pd.read_csv(DATA_DIR / "additional_250_annotated.csv", usecols=range(4))
bootstrap_records = bootstrap_rows.to_dict('records')

bootstrap_loader = cnn.make_loader(bootstrap_records, vocab, shuffle=False, tokenize_fn=preprocess)

# bootstrap opinion — reuse already-trained cnn_model_opinion
bootstrap_opinion_preds, _, _ = cnn.predict(cnn_model_opinion, bootstrap_loader, DEVICE, target=OPINION_LABEL)

# bootstrap misinfo
bootstrap_misinfo_preds, _, bootstrap_misinfo_probs = lr_transfer.run(train_rows, bootstrap_records, task=MISINFORM_LABEL)

# bootstrap combined preds
bootstrap_combined_preds = pd.DataFrame({
    "id": bootstrap_rows["id"],
    "text": bootstrap_rows["text"],
    MISINFORM_LABEL: bootstrap_misinfo_preds,
    OPINION_LABEL: bootstrap_opinion_preds
})
bootstrap_combined_preds.to_csv(DATA_DIR / "bootstrap_combined_preds.csv", index=False)

# retrain and re-evaluate opinion cnn on bootstrap-augmented data
more_train_rows = train_rows + bootstrap_combined_preds.to_dict('records')
bootstrap_train_loader = cnn.make_loader(more_train_rows, vocab, shuffle=True, tokenize_fn=preprocess)

cnn_bs_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_bs_model = cnn.train_model(cnn_bs_model, bootstrap_train_loader, dev_loader, dev_rows, DEVICE,
                               train_targets=[OPINION_LABEL])
key = f"{M_TEXT_CNN_BOOTSTRAP} — {OPINION_LABEL}"
results[key] = cnn.predict(cnn_bs_model, dev_loader, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_bs_model, result=results[key])

# retrain and re-evaluate misinformation logreg on bootstrap-augmented data
target = MISINFORM_LABEL
key = f"{M_LOGREG_EMBED_BOOTSTRAP} — {target}"
results[key] = lr_transfer.run(more_train_rows, dev_rows, task=target)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, result=results[key])



── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(250, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8735
Epoch   1 | loss=0.7049 | avg_dev_f1=0.6875 (opinion_label: 0.6875)
Epoch   2 | loss=0.5754 | avg_dev_f1=0.6909 (opinion_label: 0.6909)
Epoch   3 | loss=0.5161 | avg_dev_f1=0.7017 (opinion_label: 0.7017)
Epoch   4 | loss=0.4739 | avg_dev_f1=0.6827 (opinion_label: 0.6827)
Epoch   5 | loss=0.4212 | avg_dev_f1=0.6848 (opinion_label: 0.6848)
Epoch   6 | loss=0.3717 | avg_dev_f1=0.6970 (opinion_label: 0.6970)
Epoch   7 | loss=0.3315 | avg_dev_f1=0.7033 (opinion_label: 0.7033)
Epoch   8 | loss=0.2839 | avg_dev_f1=0.7129 (opinion_label

## Active Learning

We used the 250 bootstrap examples tagged by the model and ranked them by an active learning uncertainty score.

### Use bootstrap to collect probabilities

In [13]:

# Reuse already-trained cnn_model_opinion — no retraining needed
bs_op_preds, _, bs_op_probs = cnn.predict(cnn_model_opinion, bootstrap_loader, DEVICE, target=OPINION_LABEL)

# Reuse misinformation LogReg predictions from bootstrap cell — no retraining needed
bs_mis_preds, bs_mis_probs = bootstrap_misinfo_preds, bootstrap_misinfo_probs

# LogReg opinion predictions for committee disagreement scoring (COLX_581 Lecture 7)
bs_op_preds_lr, _, bs_op_probs_lr = lr_transfer.run(train_rows, bootstrap_records,
                                                     task=OPINION_LABEL)
bs_mis_preds_lr = bs_mis_preds

print(f"Opinion probs shape:  {np.array(bs_op_probs).shape}")
print(f"Misinfo probs shape:  {np.array(bs_mis_probs).shape}")


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(250, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7304
Opinion probs shape:  (250,)
Misinfo probs shape:  (250,)


### Uncertainty and committee disagreement scores

Combined score averages both signals

In [14]:
op_probs  = np.array(bs_op_probs,  dtype=float)
mis_probs = np.array(bs_mis_probs, dtype=float)

# Uncertainty scores
op_uncertainty  = 1.0 - np.maximum(op_probs,  1.0 - op_probs)
mis_uncertainty = 1.0 - np.maximum(mis_probs, 1.0 - mis_probs)
avg_uncertainty = (op_uncertainty + mis_uncertainty) / 2.0

# Committee disagreement: (1) CNN and LogReg prediction disagreement; (0) agreement
op_disagreement  = (np.array(bs_op_preds) != np.array(bs_op_preds_lr)).astype(float)
mis_disagreement = (np.array(bs_mis_preds) != np.array(bs_mis_preds_lr)).astype(float)
avg_disagreement = (op_disagreement + mis_disagreement) / 2.0

# Combined score (average of uncertainty and committee disagreement)
# Questioned Claude on best approach for scoring our specific dataset
combined_score = (avg_uncertainty + avg_disagreement) / 2.0

# Claude assisted with code and approach
al_df = bootstrap_rows[["id", "text"]].copy()
al_df["bs_opinion_pred"]  = bs_op_preds
al_df["bs_misinfo_pred"]  = bs_mis_preds
al_df["op_prob"]          = op_probs
al_df["mis_prob"]         = mis_probs
al_df["op_uncertainty"]   = op_uncertainty
al_df["mis_uncertainty"]  = mis_uncertainty
al_df["avg_uncertainty"]  = avg_uncertainty
al_df["op_disagreement"]  = op_disagreement
al_df["mis_disagreement"] = mis_disagreement
al_df["avg_disagreement"] = avg_disagreement
al_df["combined_score"]   = combined_score

al_df_ranked = al_df.sort_values("combined_score", ascending=False).reset_index(drop=True)
al_df_ranked.to_csv(DATA_DIR / "active_learning_ranked.csv", index=False)

print(f"Saved active_learning_ranked.csv ({len(al_df_ranked)} rows)")
print(f"\nTop 10 most uncertain/disagreed examples:")
al_df_ranked[["id", "combined_score", "avg_uncertainty", "avg_disagreement"]].head(10)

Saved active_learning_ranked.csv (250 rows)

Top 10 most uncertain/disagreed examples:


,id,combined_score,avg_uncertainty,avg_disagreement
0,30661,0.479899,0.459799,0.5
1,31917,0.461888,0.423777,0.5
2,11555,0.460118,0.420236,0.5
3,30387,0.458662,0.417323,0.5
4,16998,0.444259,0.388518,0.5
5,22373,0.438781,0.377563,0.5
6,10057,0.437904,0.375809,0.5
7,4971,0.429819,0.359638,0.5
8,40335,0.429427,0.358854,0.5
9,40994,0.427341,0.354683,0.5


### Manual Re-annotation

The file `active_learning_ranked.csv` lists all 250 bootstrap examples.

**Manual annotation task:** Re-label the top 20% by hand using the same guidelines as the original dataset. Save the result as `active_learning_annotated.csv` with columns `id`, `text`, `misinformation_label`, `opinion_label`. (Completed by Shiao-li last week).

The accumulation test below uses human labels for the top-k rows and model labels for the rest.

In [15]:
cnn_al_models = {}

#Claude assisted with fixing code errors
def run_accumulation_test(pct, al_annotated_df, bootstrap_model_df):
    """
    pct: fraction of 250 to use with human labels
    al_annotated_df: active_learning_annotated.csv (human labels in top rows)
    bootstrap_model_df: bootstrap_combined_preds.csv (model labels for all 250)
    """
    al_idx = AL_PCTS.index(pct)
    n_top = int(round(len(al_annotated_df) * pct))
    print(f"AL {pct*100:.0f}%: {n_top} human-annotated + {len(al_annotated_df)-n_top} model-labelled")

    top_k = al_annotated_df.iloc[:n_top][["id", "text",
                                          MISINFORM_LABEL, OPINION_LABEL]].copy()
    remaining_ids = set(al_annotated_df.iloc[n_top:]["id"])
    rest = bootstrap_model_df[bootstrap_model_df["id"].isin(remaining_ids)].copy()

    extra_records  = pd.concat([top_k, rest], ignore_index=True).to_dict('records')
    augmented_rows = train_rows + extra_records
    aug_loader     = cnn.make_loader(augmented_rows, vocab, shuffle=True, tokenize_fn=preprocess)

    # Retrain Opinion CNN
    model_op = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model_op = cnn.train_model(model_op, aug_loader, dev_loader,
                               dev_rows, DEVICE, train_targets=[OPINION_LABEL])
    res_op  = cnn.predict(model_op, dev_loader, DEVICE, target=OPINION_LABEL)
    m_op    = compute_metrics(*res_op)
    key_op  = f"{CNN_AL_MODELS[al_idx]} — {OPINION_LABEL}"
    results[key_op] = res_op
    metrics_cache[key_op] = m_op
    save_checkpoint(key_op, model=model_op, result=res_op)
    cnn_al_models[pct] = model_op

    # Retrain Misinformation LogReg
    res_mis = lr_transfer.run(augmented_rows, dev_rows, task=MISINFORM_LABEL)
    m_mis   = compute_metrics(*res_mis)
    key_mis = f"{LR_AL_MODELS[al_idx]} — {MISINFORM_LABEL}"
    results[key_mis] = res_mis
    metrics_cache[key_mis] = m_mis
    save_checkpoint(key_mis, result=res_mis)

    print(f"Opinion  Macro F1: {m_op['macro_f1']:.4f}  F1-op: {m_op['f1_class1']:.4f}  "
          f"AUC: {m_op.get('auc_roc', float('nan')):.4f}")
    print(f"Misinfo  Macro F1: {m_mis['macro_f1']:.4f}  F1-mis: {m_mis['f1_class1']:.4f}  "
          f"AUC: {m_mis.get('auc_roc', float('nan')):.4f}")
    return {"opinion": m_op, "misinfo": m_mis, "n_human": n_top}


### Create annotated active learning dataset

In [16]:
ranked = pd.read_csv(DATA_DIR / "active_learning_ranked.csv")
annotated = pd.read_csv(DATA_DIR / "additional_250_annotated.csv")

al_annotated = ranked.merge(
    annotated[["id", MISINFORM_LABEL, OPINION_LABEL]],
    on="id",
    how="left"
)
al_annotated.to_csv(DATA_DIR / "active_learning_annotated.csv", index=False)

### Load annotated file and run accumulation test

In [17]:
make_reproducible()

al_annotated_df    = pd.read_csv(DATA_DIR / "active_learning_annotated.csv")
bootstrap_model_df = pd.read_csv(DATA_DIR / "bootstrap_combined_preds.csv")

accum_results = {}
for pct in AL_PCTS:
    accum_results[pct] = run_accumulation_test(pct, al_annotated_df, bootstrap_model_df)


AL 5%: 12 human-annotated + 238 model-labelled
Epoch   1 | loss=0.7067 | avg_dev_f1=0.6976 (opinion_label: 0.6976)
Epoch   2 | loss=0.5808 | avg_dev_f1=0.7076 (opinion_label: 0.7076)
Epoch   3 | loss=0.5285 | avg_dev_f1=0.6948 (opinion_label: 0.6948)
Epoch   4 | loss=0.4734 | avg_dev_f1=0.6941 (opinion_label: 0.6941)
Epoch   5 | loss=0.4247 | avg_dev_f1=0.6933 (opinion_label: 0.6933)
Epoch   6 | loss=0.3772 | avg_dev_f1=0.7172 (opinion_label: 0.7172)
Epoch   7 | loss=0.3364 | avg_dev_f1=0.6849 (opinion_label: 0.6849)
Epoch   8 | loss=0.2861 | avg_dev_f1=0.7129 (opinion_label: 0.7129)
Epoch   9 | loss=0.2607 | avg_dev_f1=0.7230 (opinion_label: 0.7230)
Epoch  10 | loss=0.2119 | avg_dev_f1=0.7308 (opinion_label: 0.7308)
Epoch  11 | loss=0.2001 | avg_dev_f1=0.7172 (opinion_label: 0.7172)
Epoch  12 | loss=0.1679 | avg_dev_f1=0.7134 (opinion_label: 0.7134)
Epoch  13 | loss=0.1396 | avg_dev_f1=0.7134 (opinion_label: 0.7134)
Epoch  14 | loss=0.1239 | avg_dev_f1=0.7219 (opinion_label: 0.7219)
E

### Active learning summary table

In [18]:
summary_rows = []
for pct, res in accum_results.items():
    summary_rows.append({
        "AL threshold":      f"{pct*100:.0f}% ({res['n_human']} examples)",
        "Opinion Macro F1": res["opinion"]["macro_f1"],
        "Opinion F1 (op)":  res["opinion"]["f1_class1"],
        "Opinion AUC":      res["opinion"].get("auc_roc", float("nan")),
        "Misinfo Macro F1": res["misinfo"]["macro_f1"],
        "Misinfo F1 (mis)": res["misinfo"]["f1_class1"],
        "Misinfo AUC":      res["misinfo"].get("auc_roc", float("nan")),
    })

pd.set_option("display.float_format", "{:.4f}".format)
summary_df = pd.DataFrame(summary_rows).set_index("AL threshold")
print("Active Learning Accumulation Test")
print("─" * 70)
display(summary_df)

Active Learning Accumulation Test
──────────────────────────────────────────────────────────────────────


,Opinion Macro F1,Opinion F1 (op),Opinion AUC,Misinfo Macro F1,Misinfo F1 (mis),Misinfo AUC
AL threshold,,,,,,
5% (12 examples),0.7308,0.6971,0.7880,0.9031,0.8571,0.9476
10% (25 examples),0.7395,0.7018,0.7875,0.9031,0.8571,0.9493
15% (38 examples),0.7225,0.6961,0.7541,0.9031,0.8571,0.9471
20% (50 examples),0.7320,0.7039,0.7850,0.9031,0.8571,0.9474


### Prep and combine dataframes for augmented training set with 850 examples (boostrapping and active learning applied)

In [19]:
# Data prep to match 250 new training rows to original training set

# add Shiao-li as annotator to 250 extra training rows
annotated["annotator"] = "shiao-li"

# one-hot encode annotation tags
bin_cols = ['all_caps', 'exclamation_marks',
            'hedging', 'adjectives', 'unk']
for col in bin_cols:
    annotated[f"{col}_bin"] = annotated[col].apply(lambda x: 0 if x == '[]' else 1)

# add binarized annotator columns
annotated["jennifer"] = 0
annotated["nicole"] = 0
annotated["rachelle"] = 0
annotated["shiao-li"] = 1

# add training split column
annotated["split"] = "train"

In [20]:
bs_al_df = annotated.drop(
    columns=["misinformation_label", "opinion_label"]
    ).merge(
        al_annotated_df[["id", "misinformation_label", "opinion_label"]],
        on="id",
        how="left"
        )

training_bs_al = train_rows + bs_al_df.to_dict('records')
pd.DataFrame(training_bs_al).to_csv(DATA_DIR / "training_bs_al.csv", index=False)

## Few-Shot Learning

In [22]:
# Combine existing augmented training dataset (850 rows) with LLM produced dataset

llm_train_data = pd.read_csv(DATA_DIR / "llm_train_data.csv")
fsl_train_rows = pd.concat([pd.DataFrame(training_bs_al), pd.DataFrame(llm_train_data)], axis=0)
fsl_train_rows.to_csv(DATA_DIR / "final_train_set.csv", index=False)

#### Run models on final augmented training set with bootstrap, active learning and LLM dataset (1100 examples)

In [23]:
make_reproducible()

fsl_records = fsl_train_rows.to_dict('records')

fsl_loader = cnn.make_loader(fsl_records, vocab, shuffle=True, tokenize_fn=preprocess)

# re-train and re-evaluate TextCNN model - opinion task
cnn_fsl_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_fsl_model = cnn.train_model(cnn_fsl_model, fsl_loader, dev_loader, dev_rows, DEVICE,
                                train_targets=[OPINION_LABEL])
key = f"{M_TEXT_CNN_FSL} — {OPINION_LABEL}"
results[key] = cnn.predict(cnn_fsl_model, dev_loader, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_fsl_model, result=results[key])

# re-train and re-evaluate logreg + embeddings - misinformation task
target = MISINFORM_LABEL
key = f"{M_LOGREG_EMBED_FSL} — {target}"
results[key] = lr_transfer.run(fsl_records, dev_rows, task=target)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, result=results[key])

Epoch   1 | loss=0.6981 | avg_dev_f1=0.6643 (opinion_label: 0.6643)
Epoch   2 | loss=0.5749 | avg_dev_f1=0.7093 (opinion_label: 0.7093)
Epoch   3 | loss=0.4971 | avg_dev_f1=0.7003 (opinion_label: 0.7003)
Epoch   4 | loss=0.4541 | avg_dev_f1=0.7124 (opinion_label: 0.7124)
Epoch   5 | loss=0.4032 | avg_dev_f1=0.7112 (opinion_label: 0.7112)
Epoch   6 | loss=0.3657 | avg_dev_f1=0.7238 (opinion_label: 0.7238)
Epoch   7 | loss=0.3129 | avg_dev_f1=0.7395 (opinion_label: 0.7395)
Epoch   8 | loss=0.2705 | avg_dev_f1=0.7268 (opinion_label: 0.7268)
Epoch   9 | loss=0.2369 | avg_dev_f1=0.7118 (opinion_label: 0.7118)
Epoch  10 | loss=0.2013 | avg_dev_f1=0.7144 (opinion_label: 0.7144)
Epoch  11 | loss=0.1674 | avg_dev_f1=0.7064 (opinion_label: 0.7064)
Epoch  12 | loss=0.1471 | avg_dev_f1=0.7229 (opinion_label: 0.7229)
  Early stopping (no improvement for 5 epochs). Best F1: 0.7395
  [saved] TextCNN_FSL_—_opinion_label (state_dict, result)

── Logistic Regression  [misinformation_label] ──
  Building

### Ensembles


In [24]:
import importlib
import simple_ensemble, motivated_ensemble
importlib.reload(simple_ensemble)
importlib.reload(motivated_ensemble)
from simple_ensemble import soft_vote
from motivated_ensemble import motivated_soft_vote

# Ensemble soft vote: best LogReg + best CNN per task (selected dynamically from metrics_cache)
best_cnn = {
    task: max(CNN_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}
best_logreg = {
    task: max(LR_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}

for task in TARGETS:
    print(f"{task}: best LogReg = {best_logreg[task]} | best CNN = {best_cnn[task]}")

for task in TARGETS:
    preds, labels, probs = soft_vote([
        results[f"{best_logreg[task]} — {task}"],
        results[f"{best_cnn[task]} — {task}"],
    ])
    key = f"{M_ENSEMBLE_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_SOFT_VOTE)

opinion_label: best LogReg = LogReg (+ embeddings) | best CNN = TextCNN
misinformation_label: best LogReg = LogReg (+emb) AL 10% | best CNN = TextCNN MTL (POS + linguistic)
  [saved] Soft_Vote_Ensemble_—_opinion_label (result)
  [saved] Soft_Vote_Ensemble_—_misinformation_label (result)


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Soft Vote Ensemble — opinion_label,0.7613,0.7911,0.7314,0.7462,0.8027
Soft Vote Ensemble — misinformation_label,0.9171,0.9556,0.8785,0.8881,0.9647


In [25]:
# Motivated ensemble: best LogReg + best CNN, F1-weighted with threshold sweep
for task in TARGETS:
    ensemble_models = [best_logreg[task], best_cnn[task]]
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in ensemble_models]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in ensemble_models],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_MOTIVATED)



  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.6784      0.6559      0.7009      0.6800
        0.35      0.6995      0.6875      0.7115      0.7000
        0.40      0.7300      0.7327      0.7273      0.7300
        0.45      0.7391      0.7547      0.7234      0.7400
        0.50      0.7613      0.7911      0.7314      0.7650 ◄
        0.55      0.7141      0.7699      0.6584      0.7250
        0.60      0.6968      0.7711      0.6225      0.7150
        0.65      0.6962      0.7812      0.6111      0.7200
        0.70      0.6564      0.7715      0.5414      0.6950

  Selected threshold: 0.50
  [saved] Motivated_Ensemble_—_opinion_label (result)

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.8798      0.9291     

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble — opinion_label,0.7613,0.7911,0.7314,0.7462,0.8031
Motivated Ensemble — misinformation_label,0.9180,0.9553,0.8807,0.8966,0.9644


### All Results by Target

In [26]:
# All results by target — everything is now in metrics_cache
all_rows = [{"Model": name, "Macro F1": m["macro_f1"],
             "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
             "F1.5 (recall-weighted)": m["fbeta_class1"],
             "AUC-ROC": m.get("auc_roc", float("nan"))}
            for name, m in metrics_cache.items()]

all_df = pd.DataFrame(all_rows)

for task, label in [("Opinion", OPINION_LABEL), ("Misinformation", MISINFORM_LABEL)]:
    mask = all_df["Model"].str.endswith(f"— {label}")
    df = (all_df[mask]
          .copy()
          .assign(Model=lambda d: d["Model"].str.replace(f" — {label}", "", regex=False))
          .set_index("Model")
          .sort_values(["Macro F1", "AUC-ROC"], ascending=False))
    print(task, "Ordered by Macro F1 (Desc), AUC-ROC (Desc)")
    print("─" * 60)
    display(df)
    print()

Opinion Ordered by Macro F1 (Desc), AUC-ROC (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble,0.7613,0.7911,0.7314,0.7462,0.8031
Soft Vote Ensemble,0.7613,0.7911,0.7314,0.7462,0.8027
TextCNN,0.7410,0.7733,0.7086,0.7229,0.7992
TextCNN AL 10%,0.7395,0.7773,0.7018,0.7097,0.7875
TextCNN FSL,0.7395,0.7773,0.7018,0.7097,0.7808
TextCNN MTL (POS),0.7383,0.7593,0.7174,0.7454,0.7852
TextCNN MTL (POS + linguistic),0.7374,0.7636,0.7111,0.7330,0.7788
TextCNN AL 20%,0.7320,0.7602,0.7039,0.7241,0.7850
TextCNN Transfer,0.7314,0.7623,0.7006,0.7177,0.7827



Misinformation Ordered by Macro F1 (Desc), AUC-ROC (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble,0.9180,0.9553,0.8807,0.8966,0.9644
Soft Vote Ensemble,0.9171,0.9556,0.8785,0.8881,0.9647
LogReg (+emb) AL 10%,0.9031,0.9492,0.8571,0.8603,0.9493
LogReg (+emb) AL 5%,0.9031,0.9492,0.8571,0.8603,0.9476
LogReg (+emb) AL 20%,0.9031,0.9492,0.8571,0.8603,0.9474
LogReg (+emb) AL 15%,0.9031,0.9492,0.8571,0.8603,0.9471
LogReg (+ embeddings) Bootstrap,0.8973,0.9456,0.8491,0.8553,0.9474
LogReg (+ embeddings),0.8889,0.9428,0.8350,0.8318,0.9501
LogReg (+ embeddings) FSL,0.8859,0.9435,0.8283,0.8125,0.9693


### Best Model Analysis: Confusion Matrix & Examples

In [27]:
def print_quadrant_examples(dev_rows, preds, labels, n=3):
    """Print up to n examples from each confusion matrix quadrant."""
    quadrants = {
        "True Positives  (predicted=1, actual=1)": [],
        "True Negatives  (predicted=0, actual=0)": [],
        "False Positives (predicted=1, actual=0)": [],
        "False Negatives (predicted=0, actual=1)": [],
    }
    for row, p, l in zip(dev_rows, preds, labels):
        if   p == 1 and l == 1: quadrants["True Positives  (predicted=1, actual=1)"].append(row)
        elif p == 0 and l == 0: quadrants["True Negatives  (predicted=0, actual=0)"].append(row)
        elif p == 1 and l == 0: quadrants["False Positives (predicted=1, actual=0)"].append(row)
        elif p == 0 and l == 1: quadrants["False Negatives (predicted=0, actual=1)"].append(row)

    for label, rows in quadrants.items():
        print(f"\n── {label} ({len(rows)} total, showing {min(n, len(rows))}) ──")
        for r in rows[:n]:
            print(f"  [{r['id']}] {r['text'][:140]!r}")


for task, label in [("Opinion", OPINION_LABEL), ("Misinformation", MISINFORM_LABEL)]:
    task_keys = [k for k in metrics_cache if k.endswith(f"— {label}")]
    best_key  = max(task_keys, key=rank_key)
    preds, labels_list, probs = results[best_key]
    m = metrics_cache[best_key]

    print(f"\n{'='*60}")
    print(f"{task} — best model: {best_key.replace(f' — {label}', '')}")
    print(f"Macro F1: {m['macro_f1']:.4f}  |  AUC-ROC: {m.get('auc_roc', float('nan')):.4f}")
    print(f"{'='*60}")
    print_confusion_matrix(preds, labels_list)
    print_quadrant_examples(dev_rows, preds, labels_list, n=8)


Opinion — best model: Motivated Ensemble
Macro F1: 0.7613  |  AUC-ROC: 0.8031
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    89      28
  true=1 (opinion):   19      64

── True Positives  (predicted=1, actual=1) (64 total, showing 8) ──
  [3] 'im praying for all of my friends down in the Caribbean who have no where else to go and are forced to ride through the hurricane. be strong '
  [7] 'The aftermath of a hurricane is horrific. The heat/humidity is excruciating, no water/ice, no bathing, complete darkness, bugs, no warm food'
  [10] "@john19071969 It's unwise to flood the food supply and environment with new plant varieties. Good science requires more prudence."
  [73] 'Irony just died a thousand deaths! ???? http://t.co/dBU30ObDxz'
  [121] 'Freshman: I wish Hurricane Dorian would come our way.\n\nMe, a senior: we had Harvey two years ago.\n\nFish: but no school\n\nMe: we flooded for d'
  [131] 'As you know that Covid 19 has spr